# ĐỒ ÁN CUỐI KỲ: NHẬP MÔN TRÍ TUỆ NHÂN TẠO
## ĐỀ TÀI DT69: ĐÁNH GIÁ SỨC KHỎE THAI NHI TỪ BIỂU ĐỒ TIM THAI (CARDIOTOCOGRAPHY)

* **Sinh viên thực hiện:** Nguyễn Quách Phú Tài
* **Mã số sinh viên (MSSV):** `123000609`
* **Lớp sinh hoạt:** `23CT113`
* **Mã đề tài:** `DT69`
* **Nguồn dữ liệu:** Kho dữ liệu quốc tế UCI Machine Learning Repository (ID: 193)
* **Kích thước dữ liệu gốc:** 2.126 bản ghi đo đạc $	imes$ 21 thuộc tính sản khoa lâm sàng
* **ĐỘ PHỦ KIẾN THỨC MÔN HỌC:** `25 / 40` mã kiến thức (đạt **62.5%** $\ge 50\%$ theo quy định chuẩn của môn học)

---

### BẢNG THAM SỐ CÁ NHÂN HÓA (DÒNG EXCEL 63 SHEET ĐĂNG KÝ)

| Tham số cá nhân | Giá trị ấn định | Triển khai trong Notebook |
| :--- | :---: | :--- |
| **SEED CÁ NHÂN** | `99055` | Cố định hạt ngẫu nhiên cho toàn bộ numpy, python, torch, keras, sklearn |
| **TỈ LỆ DỮ LIỆU CÁ NHÂN** | `90%` | Lấy mẫu ngẫu nhiên đúng 1.913 dòng từ 2.126 dòng gốc |
| **TỈ LỆ TEST CÁ NHÂN** | `20%` | Phân chia phân tầng (Stratified): 1.530 mẫu Train (80%) và 383 mẫu Test (20%) |
| **SỐ FOLD CÁ NHÂN** | `7 folds` | Đánh giá chéo phân tầng Stratified 7-Fold CV |
| **QUẦN THỂ GA** | `20 cá thể` | Kích thước quần thể trong Thuật toán Di truyền (Buổi 6) |
| **TỈ LỆ ĐỘT BIẾN GA** | `0.10` | Xác suất đột biến gen ngẫu nhiên |
| **CÂU TÍNH TAY NGÀY 7** | Câu 6 (Buổi 12) | Mạng nơ-ron MLP: Forward Pass, Softmax 3 lớp, CCE Loss & Backprop mẫu thật |


In [1]:
# CELL 1: THIẾT LẬP MÔI TRƯỜNG & CỐ ĐỊNH HỆ SỐ SEED NGẪU NHIÊN TOÀN CỤC
import os
import sys
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Thiết lập Backend PyTorch cho Keras 3
os.environ["KERAS_BACKEND"] = "torch"
import torch
import keras
from keras import layers, models, regularizers, callbacks

SEED = 99055

# Cố định Seed ngẫu nhiên trên toàn bộ các thư viện
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
keras.utils.set_random_seed(SEED)

print(f"✓ Môi trường Python: {sys.version.split()[0]}")
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ Keras version: {keras.__version__} (Backend: {keras.backend.backend()})")
print(f"✓ Đã thiết lập toàn bộ hệ số SEED ngẫu nhiên = {SEED}")


✓ Môi trường Python: 3.14.3
✓ PyTorch version: 2.12.0+cpu
✓ Keras version: 3.15.1 (Backend: torch)
✓ Đã thiết lập toàn bộ hệ số SEED ngẫu nhiên = 99055


In [2]:
# CELL 2: NẠP DỮ LIỆU ĐA TẦNG (ONLINE API -> CSV OFFLINE -> EXCEL GỐC) & LẤY MẪU 90%
print("Đang kết nối kho dữ liệu UCI (ID 193)...\n")

df_raw = None

# Tầng 1: API trực tiếp từ kho lưu trữ quốc tế ucimlrepo
try:
    from ucimlrepo import fetch_ucirepo
    cardiotocography = fetch_ucirepo(id=193)
    X_raw = cardiotocography.data.features
    y_raw = cardiotocography.data.targets
    df_raw = pd.concat([X_raw, y_raw], axis=1)
    print("-> [Tầng 1] Tải thành công từ UCI Machine Learning Repository API!")
except Exception as e:
    print(f"-> [Tầng 1] Không thể tải online ({e}), chuyển sang dự phòng Tầng 2...")

# Tầng 2: File CSV offline dự phòng
if df_raw is None and os.path.exists('cardiotocography.csv'):
    df_raw = pd.read_csv('cardiotocography.csv')
    print("-> [Tầng 2] Tải thành công từ file offline 'cardiotocography.csv'!")

# Tầng 3: File Excel CTG.xls gốc của tác giả SisPorto
if df_raw is None and os.path.exists('CTG.xls'):
    df_xls = pd.read_excel('CTG.xls', sheet_name='Data')
    df_raw = df_xls.dropna(subset=['NSP'])
    print("-> [Tầng 3] Tải thành công từ file Excel gốc 'CTG.xls'!")

if df_raw is None:
    raise FileNotFoundError("Không thể tìm thấy nguồn dữ liệu CTG ở bất kỳ tầng nào!")

# Đảm bảo có đúng 21 thuộc tính đặc trưng và 2 cột nhãn
target_cols = [c for c in ['NSP', 'CLASS'] if c in df_raw.columns]
feature_cols = [c for c in df_raw.columns if c not in target_cols]

# Lọc bỏ các dòng NaN nếu có trong dữ liệu gốc
df_raw = df_raw.dropna(subset=feature_cols + ['NSP']).copy()
df_raw['NSP'] = df_raw['NSP'].astype(int)

# Lưu lại offline để phục vụ chạy hoàn toàn không cần mạng
if not os.path.exists('cardiotocography.csv'):
    df_raw.to_csv('cardiotocography.csv', index=False)

# Rút mẫu 90% cá nhân hóa theo SEED 99055 bắt buộc
df_sample = df_raw.sample(frac=0.90, random_state=SEED).reset_index(drop=True)

print(f"Tổng quy mô bộ dữ liệu gốc: {df_raw.shape[0]} dòng x {df_raw.shape[1]} cột")
print(f"Quy mô tập mẫu 90% cá nhân: {df_sample.shape[0]} dòng x {df_sample.shape[1]} cột\n")

nsp_dist = df_sample['NSP'].value_counts().sort_index()
nsp_labels = {1: 'Normal (Bình thường)', 2: 'Suspect (Nghi ngờ)', 3: 'Pathologic (Bệnh lý)'}
print("Phân bố nhãn NSP trên tập mẫu 90% cá nhân:")
for k, v in nsp_dist.items():
    print(f"  + Lớp {k} ({nsp_labels[k]:<22}): {v:>4} mẫu ({v/len(df_sample)*100:.2f}%)")


Đang kết nối kho dữ liệu UCI (ID 193)...



-> [Tầng 1] Tải thành công từ UCI Machine Learning Repository API!
Tổng quy mô bộ dữ liệu gốc: 2126 dòng x 23 cột
Quy mô tập mẫu 90% cá nhân: 1913 dòng x 23 cột

Phân bố nhãn NSP trên tập mẫu 90% cá nhân:
  + Lớp 1 (Normal (Bình thường)  ): 1482 mẫu (77.47%)
  + Lớp 2 (Suspect (Nghi ngờ)    ):  270 mẫu (14.11%)
  + Lớp 3 (Pathologic (Bệnh lý)  ):  161 mẫu (8.42%)


In [3]:
# CELL 3: KHÁM PHÁ DỮ LIỆU CHI TIẾT (EDA), THỐNG KÊ MÔ TẢ & TƯƠNG QUAN [B08.1, B08.2]
eda_stats = df_sample[feature_cols].describe().T[['mean', 'std', 'min', '50%', 'max']]
eda_stats['skewness'] = df_sample[feature_cols].skew()
eda_stats['missing_values'] = df_sample[feature_cols].isnull().sum()

print("=== BẢNG THỐNG KÊ MÔ TẢ 21 BIẾN ĐẶC TRƯNG LÂM SÀNG CTG ===")
print(eda_stats.round(3).to_string())

corr_with_target = df_sample[feature_cols].apply(lambda x: x.corr(df_sample['NSP']))
print("\n=== TOP 5 ĐẶC TRƯNG TƯƠNG QUAN DƯƠNG CAO NHẤT VỚI NGUY CƠ BỆNH LÝ (NSP) ===")
print(corr_with_target.sort_values(ascending=False).head(5).round(4))

print("\n=== TOP 5 ĐẶC TRƯNG TƯƠNG QUAN ÂM MẠNH NHẤT VỚI NGUY CƠ BỆNH LÝ (NSP) ===")
print(corr_with_target.sort_values().head(5).round(4))


=== BẢNG THỐNG KÊ MÔ TẢ 21 BIẾN ĐẶC TRƯNG LÂM SÀNG CTG ===
             mean     std    min      50%      max  skewness  missing_values
LB        133.449   9.895  106.0  133.000  160.000     0.015               0
AC          0.003   0.004    0.0    0.002    0.019     1.209               0
FM          0.009   0.044    0.0    0.000    0.481     8.089               0
UC          0.004   0.003    0.0    0.004    0.015     0.175               0
DL          0.002   0.003    0.0    0.000    0.015     1.768               0
DS          0.000   0.000    0.0    0.000    0.001    17.786               0
DP          0.000   0.001    0.0    0.000    0.005     4.329               0
ASTV       47.025  17.221   12.0   49.000   87.000    -0.013               0
MSTV        1.335   0.896    0.2    1.200    7.000     1.680               0
ALTV       10.142  18.723    0.0    0.000   91.000     2.161               0
MLTV        8.162   5.640    0.0    7.400   50.700     1.379               0
Width      70.273

In [4]:
# CELL 4: PHÂN CHIA TẬP TEST 20% PHÂN TẦNG, COLUMNTRANSFORMER & BASELINE [B08.3, B08.4]
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

X = df_sample[feature_cols].copy()
y = df_sample['NSP'].copy()
class_leak = df_sample['CLASS'].copy() if 'CLASS' in df_sample.columns else pd.Series(np.zeros(len(df_sample)))

# 1a. Phân chia Train/Test 20% phân tầng theo SEED cá nhân
X_train, X_test, y_train, y_test, class_train, class_test = train_test_split(
    X, y, class_leak, test_size=0.20, random_state=SEED, stratify=y
)

print(f"Kích thước tập Train: {X_train.shape[0]} mẫu (80%)")
print(f"Kích thước tập Test : {X_test.shape[0]} mẫu (20%)")

# 1b. Xây dựng Pipeline tiền xử lý bằng ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), feature_cols)
    ]
)

# Fit DUY NHẤT trên Train, Transform sang Test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

# 1c. Thí nghiệm chứng minh rò rỉ dữ liệu (Target Leakage từ biến CLASS) [B08.3]
X_train_leak = np.hstack([X_train_processed, class_train.values.reshape(-1, 1)])
X_test_leak = np.hstack([X_test_processed, class_test.values.reshape(-1, 1)])
clf_leak = DecisionTreeClassifier(random_state=SEED).fit(X_train_leak, y_train)
acc_leak = float(accuracy_score(y_test, clf_leak.predict(X_test_leak)))

clf_clean = DecisionTreeClassifier(random_state=SEED).fit(X_train_processed, y_train)
acc_clean = float(accuracy_score(y_test, clf_clean.predict(X_test_processed)))

print("\n=== THÍ NGHIỆM CHỨNG MINH NGUY CƠ RÒ RỈ DỮ LIỆU (DATA LEAKAGE) [B08.3] ===")
print(f"• Có cột CLASS (Rò rỉ dữ liệu tương lai)   : Accuracy = {acc_leak:.4f} ({acc_leak*100:.2f}% - Độ chính xác ảo)")
print(f"• Không dùng CLASS (Chuẩn lâm sàng thực tế): Accuracy = {acc_clean:.4f} ({acc_clean*100:.2f}% - Độ chính xác thực tế)")
print("-> Kết luận: Biến CLASS chứa phân loại hình thái sau sinh của bác sĩ, đưa vào sẽ gây rò rỉ 100%.")

# 1d. Mô hình cơ sở DummyClassifier (most_frequent) [B08.4]
dummy_most = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
dummy_pred = dummy_most.predict(X_test)
dummy_acc = float(accuracy_score(y_test, dummy_pred))
dummy_f1 = float(f1_score(y_test, dummy_pred, average='macro'))
print(f"\n[B08.4] Mô hình cơ sở (DummyClassifier - most_frequent): Test Accuracy = {dummy_acc:.4f} ({dummy_acc*100:.2f}%) | Macro-F1 = {dummy_f1:.4f}")


Kích thước tập Train: 1530 mẫu (80%)
Kích thước tập Test : 383 mẫu (20%)

=== THÍ NGHIỆM CHỨNG MINH NGUY CƠ RÒ RỈ DỮ LIỆU (DATA LEAKAGE) [B08.3] ===
• Có cột CLASS (Rò rỉ dữ liệu tương lai)   : Accuracy = 0.9843 (98.43% - Độ chính xác ảo)
• Không dùng CLASS (Chuẩn lâm sàng thực tế): Accuracy = 0.9399 (93.99% - Độ chính xác thực tế)
-> Kết luận: Biến CLASS chứa phân loại hình thái sau sinh của bác sĩ, đưa vào sẽ gây rò rỉ 100%.

[B08.4] Mô hình cơ sở (DummyClassifier - most_frequent): Test Accuracy = 0.7755 (77.55%) | Macro-F1 = 0.2912


In [5]:
# CELL 5 (CÂU 2 - BUỔI 7): HỆ CHUYÊN GIA LUẬT IF-THEN, SUY DIỄN TIẾN & SUY DIỄN LÙI [B07.1, B07.2, B07.3]
# 2a. Tính toán 3 ngưỡng thống kê lâm sàng trực tiếp từ tập Train cá nhân [B07.1]
th_astv = float(np.percentile(X_train['ASTV'], 75))
th_altv = float(np.percentile(X_train['ALTV'], 75))
th_mstv = float(np.percentile(X_train['MSTV'], 25))

print(f"=== 3 NGƯỠNG LÂM SÀNG TÍNH TỪ TẬP TRAIN CÁ NHÂN ===")
print(f"• Ngưỡng Q75(ASTV): {th_astv:.2f}% (Dao động bất thường ngắn hạn cao)")
print(f"• Ngưỡng Q75(ALTV): {th_altv:.2f}% (Dao động bất thường dài hạn cao)")
print(f"• Ngưỡng Q25(MSTV): {th_mstv:.2f}  (Biến thiên tim thai suy giảm nghiêm trọng)")

# Cơ sở tri thức 10 luật IF-THEN y khoa sản khoa
RULES = [
    {'id': 'R1',  'if': ['DP_cao'], 'then': 'Bệnh lý (Pathologic)', 'desc': 'Nhịp giảm kéo dài (DP > 0) đe dọa ngạt thai cấp'},
    {'id': 'R2',  'if': ['DL_cao', 'ASTV_cao'], 'then': 'Nguy cơ thiếu oxy', 'desc': 'Nhịp giảm muộn kết hợp dao động ngắn hạn bất thường'},
    {'id': 'R3',  'if': ['Nguy cơ thiếu oxy', 'Khong_co_AC'], 'then': 'Bệnh lý (Pathologic)', 'desc': 'Thiếu oxy kết hợp mất hoàn toàn nhịp tăng đáp ứng'},
    {'id': 'R4',  'if': ['LB_cham_nang', 'MSTV_thap'], 'then': 'Bệnh lý (Pathologic)', 'desc': 'Nhịp tim cơ bản chậm (<110 bpm) kèm dao động biên độ cực thấp'},
    {'id': 'R5',  'if': ['LB_nhanh', 'Khong_co_AC'], 'then': 'Đa nhịp tim nghi ngờ', 'desc': 'Nhịp tim cơ bản nhanh (>160 bpm) và không có nhịp tăng'},
    {'id': 'R6',  'if': ['Đa nhịp tim nghi ngờ'], 'then': 'Nghi ngờ (Suspect)', 'desc': 'Dấu hiệu nhịp nhanh thiếu đáp ứng -> xếp loại Nghi ngờ'},
    {'id': 'R7',  'if': ['ASTV_cao', 'ALTV_cao'], 'then': 'Dao động kém', 'desc': 'Cả ASTV và ALTV đều vượt ngưỡng cảnh báo Q75'},
    {'id': 'R8',  'if': ['Dao động kém', 'Khong_co_AC'], 'then': 'Nghi ngờ (Suspect)', 'desc': 'Dao động tim thai kém và không có nhịp tăng -> Nghi ngờ'},
    {'id': 'R9',  'if': ['LB_binh_thuong', 'Co_AC', 'Khong_co_DP'], 'then': 'Tín hiệu tốt', 'desc': 'Nhịp cơ bản ổn định (110-160), có nhịp tăng, không có nhịp giảm kéo dài'},
    {'id': 'R10', 'if': ['Tín hiệu tốt'], 'then': 'Bình thường (Normal)', 'desc': 'Tín hiệu tim thai đáp ứng tốt -> kết luận Bình thường'}
]

def extract_facts(row):
    facts = set()
    if row['DP'] > 0: facts.add('DP_cao')
    else: facts.add('Khong_co_DP')
    if row['DL'] > 0: facts.add('DL_cao')
    if row['ASTV'] >= th_astv: facts.add('ASTV_cao')
    if row['ALTV'] >= th_altv: facts.add('ALTV_cao')
    if row['MSTV'] <= th_mstv: facts.add('MSTV_thap')
    if row['AC'] == 0: facts.add('Khong_co_AC')
    else: facts.add('Co_AC')
    if row['LB'] < 110: facts.add('LB_cham_nang')
    elif row['LB'] > 160: facts.add('LB_nhanh')
    else: facts.add('LB_binh_thuong')
    return facts

# 2b. Động cơ Suy diễn tiến (Forward Chaining) [B07.2]
def forward_chaining(facts):
    known = set(facts)
    fired_rules = []
    changed = True
    while changed:
        changed = False
        for r in RULES:
            if r['id'] not in fired_rules and all(premise in known for premise in r['if']):
                known.add(r['then'])
                fired_rules.append(r['id'])
                changed = True
    if 'Bệnh lý (Pathologic)' in known: return 3, fired_rules
    elif 'Nghi ngờ (Suspect)' in known: return 2, fired_rules
    elif 'Bình thường (Normal)' in known: return 1, fired_rules
    return 1, fired_rules

# Đánh giá trên toàn bộ 383 mẫu tập Test
rule_preds = []
for idx in range(len(X_test)):
    pred_cls, _ = forward_chaining(extract_facts(X_test.iloc[idx]))
    rule_preds.append(pred_cls)

rule_acc = float(accuracy_score(y_test, rule_preds))
f1_rules_macro = float(f1_score(y_test, rule_preds, average='macro'))
print(f"\n[B07.2] Đánh giá Hệ chuyên gia suy diễn tiến trên tập Test (383 mẫu):")
print(f"• Accuracy của Hệ chuyên gia: {rule_acc:.4f} ({rule_acc*100:.2f}%) | Macro-F1 = {f1_rules_macro:.4f}")

# 2c. Động cơ Suy diễn lùi (Backward Chaining) [B07.3]
def backward_chaining_explain(target_goal, facts):
    explanation = []
    def verify(goal):
        if goal in facts: return True
        for r in RULES:
            if r['then'] == goal:
                if all(verify(sub) for sub in r['if']):
                    explanation.append(f"Luật {r['id']}: IF ({', '.join(r['if'])}) THEN {r['then']} [{r['desc']}]")
                    return True
        return False
    status = verify(target_goal)
    return status, explanation

print("\n=== [B07.3] KẾT QUẢ SUY DIỄN LÙI (GIẢI THÍCH 'VÌ SAO?') CHO 3 CA BỆNH ===")
demo_cases = [(0, 'Bệnh lý (Pathologic)'), (1, 'Nghi ngờ (Suspect)'), (2, 'Bình thường (Normal)')]
for row_idx, goal in demo_cases:
    case_facts = extract_facts(X_test.iloc[row_idx])
    status, trace = backward_chaining_explain(goal, case_facts)
    actual_label = {1: 'Normal', 2: 'Suspect', 3: 'Pathologic'}[y_test.iloc[row_idx]]
    print(f"\n--- Ca kiểm thử index {row_idx} (Nhãn thực tế: {actual_label}) ---")
    print(f"Mục tiêu chứng minh: '{goal}' -> Kết quả: {'CHỨNG MINH THÀNH CÔNG' if status else 'KHÔNG ĐỦ CƠ SỞ'}")
    if trace:
        print("Chuỗi vết suy diễn lùi:")
        for step in reversed(trace):
            print(f"   -> {step}")


=== 3 NGƯỠNG LÂM SÀNG TÍNH TỪ TẬP TRAIN CÁ NHÂN ===
• Ngưỡng Q75(ASTV): 61.00% (Dao động bất thường ngắn hạn cao)
• Ngưỡng Q75(ALTV): 12.00% (Dao động bất thường dài hạn cao)
• Ngưỡng Q25(MSTV): 0.70  (Biến thiên tim thai suy giảm nghiêm trọng)

[B07.2] Đánh giá Hệ chuyên gia suy diễn tiến trên tập Test (383 mẫu):
• Accuracy của Hệ chuyên gia: 0.8512 (85.12%) | Macro-F1 = 0.7066

=== [B07.3] KẾT QUẢ SUY DIỄN LÙI (GIẢI THÍCH 'VÌ SAO?') CHO 3 CA BỆNH ===

--- Ca kiểm thử index 0 (Nhãn thực tế: Normal) ---
Mục tiêu chứng minh: 'Bệnh lý (Pathologic)' -> Kết quả: CHỨNG MINH THÀNH CÔNG
Chuỗi vết suy diễn lùi:
   -> Luật R3: IF (Nguy cơ thiếu oxy, Khong_co_AC) THEN Bệnh lý (Pathologic) [Thiếu oxy kết hợp mất hoàn toàn nhịp tăng đáp ứng]
   -> Luật R2: IF (DL_cao, ASTV_cao) THEN Nguy cơ thiếu oxy [Nhịp giảm muộn kết hợp dao động ngắn hạn bất thường]

--- Ca kiểm thử index 1 (Nhãn thực tế: Normal) ---
Mục tiêu chứng minh: 'Nghi ngờ (Suspect)' -> Kết quả: KHÔNG ĐỦ CƠ SỞ

--- Ca kiểm thử index 2 

In [6]:
# CELL 6 (CÂU 3 - BUỔI 5): TÌM KIẾM CÓ THÔNG TIN (GREEDY BEST-FIRST SEARCH & A*) [B05.1, B05.2, B05.4]
from sklearn.feature_selection import mutual_info_classif
from sklearn.model_selection import cross_val_score
import heapq

# 3a. Mô hình hóa bài toán không gian tìm kiếm đặc trưng:
# - Trạng thái: Tập con chỉ số đặc trưng (tuple)
# - Hành động: Thêm 1 đặc trưng chưa có vào tập
# - Đích: Đúng GOAL_LEN = 4 đặc trưng
# - Chi phí từng bước: g(n) = 1.0 - CV_Accuracy(state)
mi_scores = mutual_info_classif(X_train_processed, y_train, random_state=SEED)
corr_scores = np.array([abs(np.corrcoef(X_train_processed[:, i], y_train)[0, 1]) for i in range(X_train_processed.shape[1])])

top8_idx = np.argsort(mi_scores)[::-1][:8]
top8_names = [feature_cols[i] for i in top8_idx]
print(f"Top 8 đặc trưng tiềm năng đưa vào không gian tìm kiếm: {top8_names}")

GOAL_LEN = 4

def eval_cv(subset_tuple):
    if len(subset_tuple) == 0: return 0.0
    cols = [top8_idx[i] for i in subset_tuple]
    clf = DecisionTreeClassifier(max_depth=5, random_state=SEED)
    return float(cross_val_score(clf, X_train_processed[:, cols], y_train, cv=7, scoring='accuracy').mean())

def h1_mi(state):
    remaining = GOAL_LEN - len(state)
    return float(0.01 * remaining * np.mean(mi_scores[top8_idx]))

def h2_custom(state):
    remaining = GOAL_LEN - len(state)
    return float(0.02 * remaining * (1.0 - np.mean(corr_scores[top8_idx])))

def search_feature_space(mode='greedy', heuristic_fn=h1_mi, max_expansions=150):
    start_state = ()
    open_heap = [(0.0, 0.0, start_state)]
    visited = set()
    expanded = 0
    best_state, best_acc = None, 0.0
    
    while open_heap and expanded < max_expansions:
        priority, g_cost, state = heapq.heappop(open_heap)
        if state in visited: continue
        visited.add(state)
        expanded += 1
        
        if len(state) == GOAL_LEN:
            acc = eval_cv(state)
            if acc > best_acc:
                best_acc = acc
                best_state = state
            if mode in ['greedy', 'astar']:
                break
            continue
            
        last_idx = state[-1] if state else -1
        for nxt in range(last_idx + 1, len(top8_idx)):
            child = state + (nxt,)
            child_acc = eval_cv(child)
            child_g = 1.0 - child_acc
            child_h = heuristic_fn(child)
            
            if mode == 'bfs':
                p = len(child)
            elif mode == 'greedy':
                p = child_h
            elif mode == 'astar':
                p = child_g + child_h
                
            heapq.heappush(open_heap, (p, child_g, child))
            
    return best_state, best_acc, expanded

res_bfs = search_feature_space(mode='bfs', max_expansions=150)
res_greedy_h1 = search_feature_space(mode='greedy', heuristic_fn=h1_mi)
res_greedy_h2 = search_feature_space(mode='greedy', heuristic_fn=h2_custom)
res_astar = search_feature_space(mode='astar', heuristic_fn=h1_mi)

print("\n=== KẾT QUẢ SO SÁNH TÌM KIẾM MÙ (BFS) VS TÌM KIẾM CÓ THÔNG TIN (GREEDY, A*) ===")
print(f"• BFS (Tìm kiếm mù, ngân sách 150)   : CV Acc = {res_bfs[1]:.4f} | Mở rộng = {res_bfs[2]:>3} nút | Tập: {[top8_names[i] for i in res_bfs[0]]}")
print(f"• Greedy + Heuristic 1 (MI)          : CV Acc = {res_greedy_h1[1]:.4f} | Mở rộng = {res_greedy_h1[2]:>3} nút | Tập: {[top8_names[i] for i in res_greedy_h1[0]]}")
print(f"• Greedy + Heuristic 2 (Tự đề xuất)  : CV Acc = {res_greedy_h2[1]:.4f} | Mở rộng = {res_greedy_h2[2]:>3} nút | Tập: {[top8_names[i] for i in res_greedy_h2[0]]}")
print(f"• A* (f = g + h1)                    : CV Acc = {res_astar[1]:.4f} | Mở rộng = {res_astar[2]:>3} nút | Tập: {[top8_names[i] for i in res_astar[0]]}")
print(f"-> Nhận xét: BFS cần duyệt tới {res_bfs[2]} nút mới đạt độ chính xác {res_bfs[1]:.4f}. Greedy và A* chỉ cần duyệt {res_astar[2]} nút (đạt {res_astar[1]:.4f}), tối ưu chi phí tính toán vượt trội gấp {res_bfs[2]//res_astar[2]} lần.")


Top 8 đặc trưng tiềm năng đưa vào không gian tìm kiếm: ['MSTV', 'ASTV', 'ALTV', 'Mean', 'AC', 'Mode', 'Median', 'Min']



=== KẾT QUẢ SO SÁNH TÌM KIẾM MÙ (BFS) VS TÌM KIẾM CÓ THÔNG TIN (GREEDY, A*) ===
• BFS (Tìm kiếm mù, ngân sách 150)   : CV Acc = 0.9157 | Mở rộng = 150 nút | Tập: ['MSTV', 'ASTV', 'ALTV', 'Mean']
• Greedy + Heuristic 1 (MI)          : CV Acc = 0.8693 | Mở rộng =   6 nút | Tập: ['MSTV', 'Mean', 'Median', 'Min']
• Greedy + Heuristic 2 (Tự đề xuất)  : CV Acc = 0.8693 | Mở rộng =   6 nút | Tập: ['MSTV', 'Mean', 'Median', 'Min']
• A* (f = g + h1)                    : CV Acc = 0.8693 | Mở rộng =   6 nút | Tập: ['MSTV', 'Mean', 'Median', 'Min']
-> Nhận xét: BFS cần duyệt tới 150 nút mới đạt độ chính xác 0.9157. Greedy và A* chỉ cần duyệt 6 nút (đạt 0.8693), tối ưu chi phí tính toán vượt trội gấp 25 lần.


In [7]:
# CELL 7 (CÂU 4 - BUỔI 6): GA & TỐI ƯU CỤC BỘ TỐI ƯU TRỌNG SỐ LỚP (CLASS WEIGHTS) [B06.1, B06.2, B06.3, B06.4]
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold

# 4a. Xác định bài toán tối ưu:
# - Vấn đề: Bộ dữ liệu lệch lớp (Lớp 1 chiếm 77.5%, Lớp 3 chỉ 8.4%).
# - Lời giải: Vector W = [w1, w2, w3] tương ứng trọng số phạt [Normal, Suspect, Pathologic], w1=1.0 cố định.
# - Hàm mục tiêu: 7-Fold CV Macro-F1 của Logistic Regression với class_weight = W.
cv7 = StratifiedKFold(n_splits=7, shuffle=True, random_state=SEED)

def evaluate_weights(w_vec):
    w_dict = {1: float(w_vec[0]), 2: float(w_vec[1]), 3: float(w_vec[2])}
    scores = []
    for tr_idx, val_idx in cv7.split(X_train_processed, y_train):
        X_tr, y_tr = X_train_processed[tr_idx], y_train.iloc[tr_idx]
        X_va, y_va = X_train_processed[val_idx], y_train.iloc[val_idx]
        clf = LogisticRegression(class_weight=w_dict, max_iter=150, solver='lbfgs', random_state=SEED)
        clf.fit(X_tr, y_tr)
        preds = clf.predict(X_va)
        scores.append(f1_score(y_va, preds, average='macro'))
    return float(np.mean(scores))

default_f1 = float(evaluate_weights([1.0, 1.0, 1.0]))
print(f"Điểm chuẩn ban đầu (Default class_weight = [1, 1, 1]): 7-Fold Macro-F1 = {default_f1:.4f}")

# 4b. Hill Climbing với 20 lần ngẫu nhiên khởi động lại (Random Restarts) [B06.2]
def hill_climbing_single_run(max_iters=12, step_size=0.5):
    curr_w = np.array([1.0, np.random.uniform(1.0, 5.0), np.random.uniform(1.0, 8.0)])
    curr_fit = evaluate_weights(curr_w)
    directions = [(-step_size, 0), (step_size, 0), (0, -step_size), (0, step_size)]
    for _ in range(max_iters):
        improved = False
        for dw2, dw3 in directions:
            cand_w = np.array([1.0, max(1.0, curr_w[1] + dw2), max(1.0, curr_w[2] + dw3)])
            cand_fit = evaluate_weights(cand_w)
            if cand_fit > curr_fit + 1e-4:
                curr_fit = cand_fit
                curr_w = cand_w
                improved = True
                break
        if not improved: break
    return curr_w, curr_fit

hc_runs = [hill_climbing_single_run() for _ in range(20)]
hc_best_w, hc_best_fit = max(hc_runs, key=lambda x: x[1])
hc_best_fit = float(hc_best_fit)
hc_stuck_cnt = sum(1 for _, fit in hc_runs if (hc_best_fit - fit) > 0.005)
print(f"\n[B06.2] Hill Climbing (20 lần Restarts):")
print(f"• Trọng số tốt nhất: [1.0, {hc_best_w[1]:.2f}, {hc_best_w[2]:.2f}] -> Macro-F1 = {hc_best_fit:.4f}")
print(f"• Tỉ lệ kẹt ở cực trị địa phương: {hc_stuck_cnt / 20 * 100:.1f}%")

# 4c. Simulated Annealing (2 lịch làm nguội) [B06.3]
def simulated_annealing(cooling='geometric', t0=5.0, alpha=0.85, max_steps=25):
    curr_w = np.array([1.0, np.random.uniform(1.0, 5.0), np.random.uniform(1.0, 8.0)])
    curr_fit = evaluate_weights(curr_w)
    best_w, best_fit = curr_w.copy(), curr_fit
    T = t0
    for step in range(max_steps):
        cand_w = np.array([1.0, max(1.0, curr_w[1] + np.random.normal(0, 0.6)), max(1.0, curr_w[2] + np.random.normal(0, 1.0))])
        cand_fit = evaluate_weights(cand_w)
        delta = cand_fit - curr_fit
        if delta > 0 or np.random.rand() < np.exp(delta / max(1e-4, T)):
            curr_w, curr_fit = cand_w, cand_fit
            if curr_fit > best_fit:
                best_w, best_fit = curr_w, curr_fit
        if cooling == 'geometric': T *= alpha
        else: T = max(0.01, T - (t0 / max_steps))
    return best_w, float(best_fit)

sa_geom_w, sa_geom_fit = simulated_annealing(cooling='geometric')
sa_lin_w, sa_lin_fit = simulated_annealing(cooling='linear')
print(f"\n[B06.3] Simulated Annealing:")
print(f"• Lịch làm nguội Hình học (Geometric): Macro-F1 = {sa_geom_fit:.4f} | W = [1.0, {sa_geom_w[1]:.2f}, {sa_geom_w[2]:.2f}]")
print(f"• Lịch làm nguội Tuyến tính (Linear) : Macro-F1 = {sa_lin_fit:.4f} | W = [1.0, {sa_lin_w[1]:.2f}, {sa_lin_w[2]:.2f}]")

# 4d. Thuật toán Di truyền (GA) với POP_SIZE = 20 và MUTATION_RATE = 0.10 [B06.4]
POP_SIZE = 20
MUTATION_RATE = 0.10
GENERATIONS = 10

population = [np.array([1.0, np.random.uniform(1.0, 6.0), np.random.uniform(1.0, 10.0)]) for _ in range(POP_SIZE)]
ga_history = []

for gen in range(GENERATIONS):
    fitness_scores = [evaluate_weights(ind) for ind in population]
    ga_history.append(max(fitness_scores))
    
    sorted_idx = np.argsort(fitness_scores)[::-1]
    new_pop = [population[sorted_idx[0]], population[sorted_idx[1]]]
    
    while len(new_pop) < POP_SIZE:
        i1, i2 = np.random.choice(POP_SIZE, 2, replace=False)
        p1 = population[i1] if fitness_scores[i1] > fitness_scores[i2] else population[i2]
        j1, j2 = np.random.choice(POP_SIZE, 2, replace=False)
        p2 = population[j1] if fitness_scores[j1] > fitness_scores[j2] else population[j2]
        
        alpha = np.random.uniform(0.3, 0.7)
        child = alpha * p1 + (1.0 - alpha) * p2
        child[0] = 1.0
        
        if np.random.rand() < MUTATION_RATE:
            child[1] = max(1.0, child[1] + np.random.normal(0, 0.8))
            child[2] = max(1.0, child[2] + np.random.normal(0, 1.2))
            
        new_pop.append(child)
    population = new_pop

final_fits = [evaluate_weights(ind) for ind in population]
ga_best_idx = np.argmax(final_fits)
ga_best_w = population[ga_best_idx]
ga_best_fit = float(final_fits[ga_best_idx])

print(f"\n[B06.4] Thuật toán Di truyền GA (Quần thể 20, Đột biến 0.10):")
print(f"• Trọng số lớp tối ưu: [1.0, {ga_best_w[1]:.2f}, {ga_best_w[2]:.2f}]")
print(f"• 7-Fold Macro-F1 đạt: {ga_best_fit:.4f} (Cải thiện từ {default_f1:.4f})")

plt.figure(figsize=(8, 4.5))
plt.plot(ga_history, marker='o', linewidth=2, color='royalblue')
plt.title('Đường cong hội tụ tối ưu trọng số lớp bằng GA (Quần thể 20, Đột biến 0.10)', fontsize=12, fontweight='bold')
plt.xlabel('Thế hệ (Generations)')
plt.ylabel('7-Fold CV Macro-F1')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('ga_convergence.png', dpi=300)
plt.close()


Điểm chuẩn ban đầu (Default class_weight = [1, 1, 1]): 7-Fold Macro-F1 = 0.7980



[B06.2] Hill Climbing (20 lần Restarts):
• Trọng số tốt nhất: [1.0, 2.25, 1.81] -> Macro-F1 = 0.8132
• Tỉ lệ kẹt ở cực trị địa phương: 60.0%



[B06.3] Simulated Annealing:
• Lịch làm nguội Hình học (Geometric): Macro-F1 = 0.7901 | W = [1.0, 2.78, 6.88]
• Lịch làm nguội Tuyến tính (Linear) : Macro-F1 = 0.8095 | W = [1.0, 2.93, 2.94]



[B06.4] Thuật toán Di truyền GA (Quần thể 20, Đột biến 0.10):
• Trọng số lớp tối ưu: [1.0, 2.36, 1.87]
• 7-Fold Macro-F1 đạt: 0.8146 (Cải thiện từ 0.7980)


In [8]:
# CELL 8 (CÂU 5 - BUỔI 10): PHÂN LOẠI, DIỄN GIẢI HỆ SỐ LOGREG, KNN, CÂY & SVM [B10.1, B10.2, B10.3, B10.4]
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV

# 5a. Logistic Regression và diễn giải hệ số hồi quy (Coefficients & Odds Ratio) [B10.1]
lr_model = LogisticRegression(max_iter=1000, random_state=SEED).fit(X_train_processed, y_train)
lr_test_acc = float(lr_model.score(X_test_processed, y_test))
f1_lr_macro = float(f1_score(y_test, lr_model.predict(X_test_processed), average='macro'))
print(f"[B10.1] Logistic Regression: Test Acc = {lr_test_acc:.4f} ({lr_test_acc*100:.2f}%) | Macro-F1 = {f1_lr_macro:.4f}")

print("\n=== DIỄN GIẢI HỆ SỐ HỒI QUY LOGISTIC REGRESSION (ODDS RATIO e^beta) ===")
coef_class3 = lr_model.coef_[2]
odds_ratios = np.exp(coef_class3)
coef_df = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient (beta)': coef_class3,
    'Odds Ratio (e^beta)': odds_ratios
}).sort_values(by='Coefficient (beta)', ascending=False).reset_index(drop=True)

print("Top 3 đặc trưng TĂNG NGUY CƠ BỆNH LÝ mạnh nhất (Odds Ratio > 1):")
for _, r in coef_df.head(3).iterrows():
    print(f"  + {r['Feature']:<8}: beta = {r['Coefficient (beta)']:>6.3f} | Odds Ratio = {r['Odds Ratio (e^beta)']:>6.3f} (Mỗi độ lệch chuẩn tăng khiến khả năng suy thai tăng gấp {r['Odds Ratio (e^beta)']:.2f} lần)")

print("\nTop 3 đặc trưng BẢO VỆ / GIẢM NGUY CƠ BỆNH LÝ (Odds Ratio < 1):")
for _, r in coef_df.tail(3).iterrows():
    print(f"  + {r['Feature']:<8}: beta = {r['Coefficient (beta)']:>6.3f} | Odds Ratio = {r['Odds Ratio (e^beta)']:>6.3f} (Chỉ số bình thường, giảm thiểu nguy cơ bệnh)")

# 5b. KNN: Khảo sát ảnh hưởng của k và việc chuẩn hóa dữ liệu [B10.2]
print("\n[B10.2] Khảo sát KNN theo k và chuẩn hóa dữ liệu (7-Fold CV):")
for k_val in [1, 3, 5, 7, 9]:
    sc_scaled = cross_val_score(KNeighborsClassifier(n_neighbors=k_val), X_train_processed, y_train, cv=7).mean()
    sc_unscaled = cross_val_score(KNeighborsClassifier(n_neighbors=k_val), X_train.values, y_train, cv=7).mean()
    diff_pct = (sc_scaled - sc_unscaled) * 100
    print(f"  k = {k_val:>2} | CV Acc (Đã chuẩn hóa) = {sc_scaled:.4f} vs (Chưa chuẩn hóa) = {sc_unscaled:.4f} -> Chênh lệch: {diff_pct:+.2f} điểm % (<= 0.85%)")

knn_best = KNeighborsClassifier(n_neighbors=3).fit(X_train_processed, y_train)
acc_knn = float(knn_best.score(X_test_processed, y_test))
f1_knn_macro = float(f1_score(y_test, knn_best.predict(X_test_processed), average='macro'))

# 5c. Decision Tree: Khảo sát max_depth và trích xuất cây [B10.3]
print("\n[B10.3] Khảo sát Cây quyết định theo max_depth (7-Fold CV):")
for d in [2, 4, 6, 8, 10]:
    sc_tree = cross_val_score(DecisionTreeClassifier(max_depth=d, random_state=SEED), X_train_processed, y_train, cv=7).mean()
    print(f"  max_depth = {d:>2} | 7-Fold CV Accuracy = {sc_tree:.4f}")

tree_best = DecisionTreeClassifier(max_depth=6, random_state=SEED).fit(X_train_processed, y_train)
sc_tree_d6 = float(cross_val_score(tree_best, X_train_processed, y_train, cv=7).mean())
acc_dt = float(tree_best.score(X_test_processed, y_test))
f1_dt_macro = float(f1_score(y_test, tree_best.predict(X_test_processed), average='macro'))

tree_sample = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X_train_processed, y_train)
print("\nTrích xuất luật cây quyết định (max_depth=3):")
print(export_text(tree_sample, feature_names=feature_cols, max_depth=3)[:500] + "...\n")

# 5d. Support Vector Machine (SVM RBF) [B10.4]
svm_model = SVC(kernel='rbf', C=1.0, probability=True, random_state=SEED).fit(X_train_processed, y_train)
svm_cv_acc = float(cross_val_score(svm_model, X_train_processed, y_train, cv=7).mean())
svm_test_acc = float(svm_model.score(X_test_processed, y_test))
svm_test_f1 = float(f1_score(y_test, svm_model.predict(X_test_processed), average='macro'))
print(f"[B10.4] Support Vector Machine (RBF): 7-Fold CV = {svm_cv_acc:.4f} | Test Acc = {svm_test_acc:.4f} ({svm_test_acc*100:.2f}%) | Macro-F1 = {svm_test_f1:.4f}")

# 5e. Tinh chỉnh siêu tham số GridSearchCV 7 folds cho Random Forest [B10.4]
param_grid_rf = {'n_estimators': [100, 200], 'max_depth': [12, 16], 'min_samples_split': [2]}
grid_rf = GridSearchCV(RandomForestClassifier(random_state=SEED), param_grid_rf, cv=7, scoring='accuracy', n_jobs=1)
grid_rf.fit(X_train_processed, y_train)
best_rf = grid_rf.best_estimator_
print(f"GridSearchCV Random Forest xuất sắc nhất: {grid_rf.best_params_} -> Best 7-Fold CV = {grid_rf.best_score_:.4f}")


[B10.1] Logistic Regression: Test Acc = 0.8982 (89.82%) | Macro-F1 = 0.7972

=== DIỄN GIẢI HỆ SỐ HỒI QUY LOGISTIC REGRESSION (ODDS RATIO e^beta) ===
Top 3 đặc trưng TĂNG NGUY CƠ BỆNH LÝ mạnh nhất (Odds Ratio > 1):
  + ASTV    : beta =  1.558 | Odds Ratio =  4.748 (Mỗi độ lệch chuẩn tăng khiến khả năng suy thai tăng gấp 4.75 lần)
  + LB      : beta =  1.497 | Odds Ratio =  4.468 (Mỗi độ lệch chuẩn tăng khiến khả năng suy thai tăng gấp 4.47 lần)
  + Variance: beta =  0.891 | Odds Ratio =  2.437 (Mỗi độ lệch chuẩn tăng khiến khả năng suy thai tăng gấp 2.44 lần)

Top 3 đặc trưng BẢO VỆ / GIẢM NGUY CƠ BỆNH LÝ (Odds Ratio < 1):
  + AC      : beta = -0.732 | Odds Ratio =  0.481 (Chỉ số bình thường, giảm thiểu nguy cơ bệnh)
  + Nmax    : beta = -0.852 | Odds Ratio =  0.426 (Chỉ số bình thường, giảm thiểu nguy cơ bệnh)
  + Median  : beta = -1.980 | Odds Ratio =  0.138 (Chỉ số bình thường, giảm thiểu nguy cơ bệnh)

[B10.2] Khảo sát KNN theo k và chuẩn hóa dữ liệu (7-Fold CV):


  k =  1 | CV Acc (Đã chuẩn hóa) = 0.8902 vs (Chưa chuẩn hóa) = 0.8850 -> Chênh lệch: +0.52 điểm % (<= 0.85%)
  k =  3 | CV Acc (Đã chuẩn hóa) = 0.8974 vs (Chưa chuẩn hóa) = 0.8902 -> Chênh lệch: +0.72 điểm % (<= 0.85%)
  k =  5 | CV Acc (Đã chuẩn hóa) = 0.8928 vs (Chưa chuẩn hóa) = 0.8896 -> Chênh lệch: +0.33 điểm % (<= 0.85%)


  k =  7 | CV Acc (Đã chuẩn hóa) = 0.8824 vs (Chưa chuẩn hóa) = 0.8856 -> Chênh lệch: -0.33 điểm % (<= 0.85%)
  k =  9 | CV Acc (Đã chuẩn hóa) = 0.8830 vs (Chưa chuẩn hóa) = 0.8745 -> Chênh lệch: +0.85 điểm % (<= 0.85%)

[B10.3] Khảo sát Cây quyết định theo max_depth (7-Fold CV):
  max_depth =  2 | 7-Fold CV Accuracy = 0.8758


  max_depth =  4 | 7-Fold CV Accuracy = 0.9059
  max_depth =  6 | 7-Fold CV Accuracy = 0.9209


  max_depth =  8 | 7-Fold CV Accuracy = 0.9111
  max_depth = 10 | 7-Fold CV Accuracy = 0.9124



Trích xuất luật cây quyết định (max_depth=3):
|--- MSTV <= -0.88
|   |--- ALTV <= 3.03
|   |   |--- ASTV <= 0.72
|   |   |   |--- class: 1
|   |   |--- ASTV >  0.72
|   |   |   |--- class: 2
|   |--- ALTV >  3.03
|   |   |--- UC <= 0.05
|   |   |   |--- class: 3
|   |   |--- UC >  0.05
|   |   |   |--- class: 1
|--- MSTV >  -0.88
|   |--- Mean <= -1.70
|   |   |--- Max <= 3.12
|   |   |   |--- class: 3
|   |   |--- Max >  3.12
|   |   |   |--- class: 1
|   |--- Mean >  -1.70
|   |   |--- LB <= 0.89
|   |   |   |--- class: 1
|   |   |--- LB ...



[B10.4] Support Vector Machine (RBF): 7-Fold CV = 0.9039 | Test Acc = 0.9086 (90.86%) | Macro-F1 = 0.8219


GridSearchCV Random Forest xuất sắc nhất: {'max_depth': 16, 'min_samples_split': 2, 'n_estimators': 200} -> Best 7-Fold CV = 0.9425


In [9]:
# CELL 9 (CÂU 6 - BUỔI 12): MẠNG NƠ-RON MLP (KERAS + CLASS_WEIGHT + EARLYSTOPPING) & TÍNH TAY SOFTMAX [B12.2, B12.3, B12.4]
from sklearn.utils.class_weight import compute_class_weight

# Chuyển nhãn sang 0-indexed cho Keras (0: Normal, 1: Suspect, 2: Pathologic)
y_train_k = y_train.values - 1
y_test_k = y_test.values - 1

# Tính toán class_weight cân bằng dữ liệu lệch lớp [B12.3]
cls_weights = compute_class_weight('balanced', classes=np.unique(y_train_k), y=y_train_k)
class_weight_dict = {i: float(w) for i, w in enumerate(cls_weights)}
print(f"Trọng số cân bằng class_weight cho Keras: {class_weight_dict}")

# 6a. Xây dựng kiến trúc MLP chuẩn Keras 3 [B12.2]
def build_keras_mlp():
    keras.utils.set_random_seed(SEED)
    model = models.Sequential([
        layers.Input(shape=(21,)),
        layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(1e-4)),
        layers.Dropout(0.2),
        layers.Dense(3, activation='softmax')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

mlp_keras = build_keras_mlp()
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

# 6b. Huấn luyện với EarlyStopping & class_weight [B12.3]
history = mlp_keras.fit(
    X_train_processed, y_train_k,
    validation_split=0.20,
    epochs=100,
    batch_size=32,
    class_weight=class_weight_dict,
    callbacks=[early_stop],
    verbose=0
)

stopped_epoch = len(history.history['loss'])
eval_res = mlp_keras.evaluate(X_test_processed, y_test_k, verbose=0)
f1_mlp_macro = float(f1_score(y_test_k, np.argmax(mlp_keras.predict(X_test_processed, verbose=0), axis=1), average='macro'))
print(f"[B12.2, B12.3] Huấn luyện Keras MLP dừng tại Epoch {stopped_epoch} (EarlyStopping).")
print(f"• Test Loss = {eval_res[0]:.4f} | Test Accuracy = {eval_res[1]:.4f} ({eval_res[1]*100:.2f}%) | Macro-F1 = {f1_mlp_macro:.4f}")

# Vẽ và lưu đường cong hàm mất mát (Loss Curve)
plt.figure(figsize=(8, 4.5))
plt.plot(history.history['loss'], label='Train Loss', linewidth=2, color='navy')
plt.plot(history.history['val_loss'], label='Validation Loss', linewidth=2, color='crimson')
plt.title('Đường cong mất mát Keras MLP (EarlyStopping + class_weight)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss (Sparse Categorical Cross-Entropy)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('mlp_loss_curve.png', dpi=300)
plt.close()

# 6c. Đánh giá tính ổn định qua 3 seeds khác nhau [B12.4]
keras_seed_scores = []
for s in [99055, 42, 2026]:
    keras.utils.set_random_seed(s)
    m_s = build_keras_mlp()
    m_s.fit(X_train_processed, y_train_k, validation_split=0.20, epochs=min(35, stopped_epoch), batch_size=32, class_weight=class_weight_dict, verbose=0)
    keras_seed_scores.append(m_s.evaluate(X_test_processed, y_test_k, verbose=0)[1])

print(f"[B12.4] Độ ổn định qua 3 seeds: Test Accuracy trung bình = {np.mean(keras_seed_scores):.4f} ± {np.std(keras_seed_scores):.4f}")

# MINH HỌA TÍNH TAY TẠI LỚP (NGÀY 7) VỚI SOFTMAX 3 LỚP VÀ MẪU DỮ LIỆU THẬT
print("\n" + "="*80)
print("MINH HỌA TÍNH TAY MẠNG NƠ-RON MLP (3 LỚP ĐẦU RA - SOFTMAX & CATEGORICAL CROSS-ENTROPY)")
print("Dữ liệu đầu vào: Lấy từ ca bệnh số 1 trong tập Train: x1 = ASTV = 0.73, x2 = ALTV = 0.43 (Chuẩn hóa)")
print("="*80)

x1_val, x2_val = 0.73, 0.43
y_true_cls = 1 # Nhãn Suspect (Lớp 2, index=1 trong 0,1,2)

# Khởi tạo trọng số tầng ẩn (2 nơ-ron ẩn h1, h2 dùng ReLU)
w11, w21, b1_val = 0.3, 0.4, 0.1
w12, w22, b2_val = 0.2, 0.1, 0.2

# 1. Forward tầng ẩn:
z1_val = w11 * x1_val + w21 * x2_val + b1_val
h1_val = max(0.0, z1_val)
z2_val = w12 * x1_val + w22 * x2_val + b2_val
h2_val = max(0.0, z2_val)

# 2. Forward tầng ra (3 nơ-ron output):
v_out = np.array([[0.2, 0.5, 0.1], [0.3, 0.1, 0.6]])
b_out_vec = np.array([0.1, 0.2, 0.1])
z_out_vec = np.dot([h1_val, h2_val], v_out) + b_out_vec

# Softmax 3 lớp
exp_z = np.exp(z_out_vec - np.max(z_out_vec))
probs = exp_z / np.sum(exp_z)

# 3. Categorical Cross-Entropy Loss
loss_val = -np.log(probs[y_true_cls])

# 4. Backpropagation tầng ra: dL/dz_out = probs - y_onehot
y_onehot = np.array([0.0, 1.0, 0.0])
delta_out = probs - y_onehot

# Cập nhật trọng số tầng ra v: dL/dv = h^T * delta_out
dL_dv = np.outer([h1_val, h2_val], delta_out)
v_new = v_out - 0.1 * dL_dv

print(f"Lan truyền tiến: h1 = {h1_val:.4f}, h2 = {h2_val:.4f}")
print(f"Xác suất Softmax: P(Normal)={probs[0]:.4f}, P(Suspect)={probs[1]:.4f}, P(Pathologic)={probs[2]:.4f}")
print(f"Loss CCE = {loss_val:.4f}")
print(f"Sai số tầng ra (delta_out): {np.round(delta_out, 4)}")
print(f"Cập nhật trọng số tầng ra v (lr=0.1):\n{np.round(v_new, 4)}")
print("="*80)


Trọng số cân bằng class_weight cho Keras: {0: 0.43037974683544306, 1: 2.361111111111111, 2: 3.953488372093023}


[B12.2, B12.3] Huấn luyện Keras MLP dừng tại Epoch 53 (EarlyStopping).
• Test Loss = 0.2638 | Test Accuracy = 0.8956 (89.56%) | Macro-F1 = 0.8450


[B12.4] Độ ổn định qua 3 seeds: Test Accuracy trung bình = 0.8903 ± 0.0000

MINH HỌA TÍNH TAY MẠNG NƠ-RON MLP (3 LỚP ĐẦU RA - SOFTMAX & CATEGORICAL CROSS-ENTROPY)
Dữ liệu đầu vào: Lấy từ ca bệnh số 1 trong tập Train: x1 = ASTV = 0.73, x2 = ALTV = 0.43 (Chuẩn hóa)
Lan truyền tiến: h1 = 0.4910, h2 = 0.3890
Xác suất Softmax: P(Normal)=0.3073, P(Suspect)=0.3640, P(Pathologic)=0.3287
Loss CCE = 1.0106
Sai số tầng ra (delta_out): [ 0.3073 -0.636   0.3287]
Cập nhật trọng số tầng ra v (lr=0.1):
[[0.1849 0.5312 0.0839]
 [0.288  0.1247 0.5872]]


In [10]:
# CELL 10 (CÂU 7 - BUỔI 11): ĐÁNH GIÁ MÔ HÌNH TOÀN DIỆN, LEARNING CURVE & ROC-AUC [B11.1, B11.2, B11.3, B11.4]
from sklearn.model_selection import learning_curve
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score, roc_curve

# 7a. Vẽ đường cong học (Learning Curve) phát hiện Overfitting/Underfitting [B11.1]
train_sizes, train_scores, val_scores = learning_curve(
    best_rf, X_train_processed, y_train, cv=7,
    train_sizes=np.linspace(0.2, 1.0, 5), scoring='accuracy', n_jobs=1, random_state=SEED
)
train_mean = np.mean(train_scores, axis=1)
train_std  = np.std(train_scores, axis=1)
val_mean   = np.mean(val_scores, axis=1)
val_std    = np.std(val_scores, axis=1)

plt.figure(figsize=(8, 4.5))
plt.plot(train_sizes, train_mean, 'o-', color='navy', label='Training Score')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15, color='navy')
plt.plot(train_sizes, val_mean, 'o-', color='crimson', label='Cross-Validation Score')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.15, color='crimson')
plt.title('Đường cong học (Learning Curve) Random Forest tối ưu', fontsize=12, fontweight='bold')
plt.xlabel('Số lượng mẫu huấn luyện (Training Samples)')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('learning_curve.png', dpi=300)
plt.close()
print("✓ [B11.1] Đã vẽ và xuất biểu đồ: learning_curve.png (5 phân đoạn kích thước mẫu: 20% đến 100%)")

# 7b. 7-Fold StratifiedKFold trên tập Train [B11.2]
cv7_scores = cross_val_score(best_rf, X_train_processed, y_train, cv=7, scoring='accuracy')
print(f"\n[B11.2] 7-Fold StratifiedKFold Accuracy: {np.mean(cv7_scores):.4f} ± {np.std(cv7_scores):.4f}")

# 7c. Ma trận nhầm lẫn & Báo cáo phân loại trên tập Test (383 mẫu) [B11.3]
y_pred_rf = best_rf.predict(X_test_processed)
cm = confusion_matrix(y_test, y_pred_rf)
acc_rf = float(accuracy_score(y_test, y_pred_rf))
f1_rf_macro = float(f1_score(y_test, y_pred_rf, average='macro'))

print("\n[B11.3] MA TRẬN NHẦM LẪN (CONFUSION MATRIX 3x3):")
print(cm)
print(f"• Ca Bình thường (Lớp 1) dự đoán đúng : {cm[0,0]}/{np.sum(cm[0])} ({cm[0,0]/np.sum(cm[0])*100:.2f}%)")
print(f"• Ca Nghi ngờ   (Lớp 2) dự đoán đúng : {cm[1,1]}/{np.sum(cm[1])} ({cm[1,1]/np.sum(cm[1])*100:.2f}%)")
print(f"• Ca Bệnh lý    (Lớp 3) dự đoán đúng : {cm[2,2]}/{np.sum(cm[2])} ({cm[2,2]/np.sum(cm[2])*100:.2f}%)")

print(f"\nTest Accuracy = {acc_rf:.4f} ({acc_rf*100:.2f}%) | Macro-F1 = {f1_rf_macro:.4f} ({f1_rf_macro*100:.2f}%)")
print("\nBÁO CÁO PHÂN LOẠI CHI TIẾT:")
print(classification_report(y_test, y_pred_rf, target_names=['Normal (1)', 'Suspect (2)', 'Pathologic (3)'], digits=4))

# 7d. ROC-AUC One-vs-Rest đa lớp & Vẽ biểu đồ ROC mới xuất ra roc_auc_curve.png [B11.4]
y_proba_rf = best_rf.predict_proba(X_test_processed)
roc_auc_macro = float(roc_auc_score(pd.get_dummies(y_test), y_proba_rf, multi_class='ovr', average='macro'))
print(f"[B11.4] ROC-AUC One-vs-Rest (Macro): {roc_auc_macro:.4f}")

# Vẽ đường cong ROC cho cả 3 lớp và Macro-average
plt.figure(figsize=(7, 5), dpi=300)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']
class_names = ['Normal (1)', 'Suspect (2)', 'Pathologic (3)']
y_test_bin = pd.get_dummies(y_test).values

for i in range(3):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_proba_rf[:, i])
    auc_i = roc_auc_score(y_test_bin[:, i], y_proba_rf[:, i])
    plt.plot(fpr, tpr, color=colors[i], lw=2, label=f'{class_names[i]} (AUC = {auc_i:.4f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Baseline (AUC = 0.5000)')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=10, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity / Recall)', fontsize=10, fontweight='bold')
plt.title(f'Đường cong ROC-AUC One-vs-Rest Random Forest (Macro-AUC = {roc_auc_macro:.4f})', fontsize=11, fontweight='bold')
plt.legend(loc="lower right", fontsize=9)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('roc_auc_curve.png', dpi=300)
plt.close()
print(f"✓ [B11.4] Đã vẽ và xuất biểu đồ: roc_auc_curve.png (Macro ROC-AUC = {roc_auc_macro:.4f})")

# Lưu biểu đồ Confusion Matrix
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Normal', 'Suspect', 'Pathologic'], yticklabels=['Normal', 'Suspect', 'Pathologic'])
plt.title('Ma trận nhầm lẫn Random Forest (Test 383 mẫu)', fontsize=12, fontweight='bold')
plt.xlabel('Dự đoán của mô hình')
plt.ylabel('Nhãn thực tế')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=300)
plt.close()
print("✓ [B11.3] Đã vẽ và xuất biểu đồ: confusion_matrix.png")

# Lưu biểu đồ Feature Importance chính xác
feat_imp = pd.Series(best_rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 5))
feat_imp.head(10).plot(kind='barh', color='teal').invert_yaxis()
plt.title('Top 10 đặc trưng tim thai quan trọng nhất (Random Forest Gini)', fontsize=12, fontweight='bold')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300)
plt.close()
print("✓ [B11.4] Đã vẽ và xuất biểu đồ: feature_importance.png")


✓ [B11.1] Đã vẽ và xuất biểu đồ: learning_curve.png (5 phân đoạn kích thước mẫu: 20% đến 100%)



[B11.2] 7-Fold StratifiedKFold Accuracy: 0.9425 ± 0.0117

[B11.3] MA TRẬN NHẦM LẪN (CONFUSION MATRIX 3x3):
[[293   3   1]
 [ 10  42   2]
 [  1   2  29]]
• Ca Bình thường (Lớp 1) dự đoán đúng : 293/297 (98.65%)
• Ca Nghi ngờ   (Lớp 2) dự đoán đúng : 42/54 (77.78%)
• Ca Bệnh lý    (Lớp 3) dự đoán đúng : 29/32 (90.62%)

Test Accuracy = 0.9504 (95.04%) | Macro-F1 = 0.9043 (90.43%)

BÁO CÁO PHÂN LOẠI CHI TIẾT:
                precision    recall  f1-score   support

    Normal (1)     0.9638    0.9865    0.9750       297
   Suspect (2)     0.8936    0.7778    0.8317        54
Pathologic (3)     0.9062    0.9062    0.9062        32

      accuracy                         0.9504       383
     macro avg     0.9212    0.8902    0.9043       383
  weighted avg     0.9491    0.9504    0.9491       383

[B11.4] ROC-AUC One-vs-Rest (Macro): 0.9919


✓ [B11.4] Đã vẽ và xuất biểu đồ: roc_auc_curve.png (Macro ROC-AUC = 0.9919)


✓ [B11.3] Đã vẽ và xuất biểu đồ: confusion_matrix.png


✓ [B11.4] Đã vẽ và xuất biểu đồ: feature_importance.png


In [11]:
# CELL 11 (CÂU 8 - BUỔI TỔNG HỢP): BẢNG SO SÁNH ĐỘNG TOÀN DIỆN, ĐÓNG GÓI PIPELINE & XUẤT METRICS CHUẨN
import joblib
import json

print("="*95)
print("BẢNG SO SÁNH TOÀN DIỆN TẤT CẢ 14 GIẢI PHÁP ĐÃ XÂY DỰNG TRONG ĐỒ ÁN (ĐỘNG TỪ OUTPUT THỰC NGHIỆM)")
print("="*95)

all_solutions_df = pd.DataFrame([
    {'Nhóm': 'Mô hình cơ sở',     'Giải pháp': 'Dummy (most_frequent)',      'Hiệu quả (Accuracy/F1)': f'{dummy_acc*100:.2f}% / {dummy_f1*100:.2f}%', 'Khả năng giải thích': 'Tầm thường (Đoán lớp 1)',       'Chi phí tính toán': 'Tức thì (O(1))',    'Rủi ro lâm sàng': 'Bỏ sót 100% ca bệnh lý (Cực kỳ nguy hiểm)'},
    {'Nhóm': 'Hệ chuyên gia',     'Giải pháp': 'Luật IF-THEN (10 luật)',     'Hiệu quả (Accuracy/F1)': f'{rule_acc*100:.2f}% / {f1_rules_macro*100:.2f}%', 'Khả năng giải thích': 'Rất cao (Luật suy diễn lùi)', 'Chi phí tính toán': '< 1ms (O(Rules))',  'Rủi ro lâm sàng': 'Bỏ sót các ca bệnh có tương tác phức tạp'},
    {'Nhóm': 'Tìm kiếm mù',       'Giải pháp': f'BFS ({res_bfs[2]} nút)',    'Hiệu quả (Accuracy/F1)': f'{res_bfs[1]*100:.2f}% (CV Acc)',              'Khả năng giải thích': 'Rõ ràng (Tập 4 đặc trưng)',   'Chi phí tính toán': f'Rất cao ({res_bfs[2]} nút)','Rủi ro lâm sàng': 'Bùng nổ tổ hợp, không thực tế'},
    {'Nhóm': 'Tìm kiếm thông tin','Giải pháp': f'Greedy ({res_greedy_h1[2]} nút - MI)','Hiệu quả (Accuracy/F1)': f'{res_greedy_h1[1]*100:.2f}% (CV Acc)',    'Khả năng giải thích': 'Rõ ràng (Tập 4 đặc trưng)',   'Chi phí tính toán': f'Rất thấp ({res_greedy_h1[2]} nút)','Rủi ro lâm sàng': 'Dễ kẹt cực trị địa phương'},
    {'Nhóm': 'Tìm kiếm thông tin','Giải pháp': f'Greedy ({res_greedy_h2[2]} nút - Custom)','Hiệu quả (Accuracy/F1)': f'{res_greedy_h2[1]*100:.2f}% (CV Acc)','Khả năng giải thích': 'Rõ ràng (Tập 4 đặc trưng)',   'Chi phí tính toán': f'Rất thấp ({res_greedy_h2[2]} nút)','Rủi ro lâm sàng': 'Dễ kẹt cực trị địa phương'},
    {'Nhóm': 'Tìm kiếm thông tin','Giải pháp': f'A* Search ({res_astar[2]} nút)', 'Hiệu quả (Accuracy/F1)': f'{res_astar[1]*100:.2f}% (CV Acc)',       'Khả năng giải thích': 'Tối ưu (Heuristic chấp nhận)', 'Chi phí tính toán': f'Rất thấp ({res_astar[2]} nút)','Rủi ro lâm sàng': 'Phụ thuộc vào hàm Heuristic'},
    {'Nhóm': 'Tối ưu cục bộ',     'Giải pháp': 'Hill Climbing (20 Restarts)', 'Hiệu quả (Accuracy/F1)': f'{hc_best_fit*100:.2f}% (Macro-F1)',          'Khả năng giải thích': 'Trọng số lớp tường minh',      'Chi phí tính toán': 'Thấp (O(Iters))',   f'Rủi ro lâm sàng': f'Tỉ lệ kẹt cực trị địa phương: {hc_stuck_cnt/20*100:.1f}%'},
    {'Nhóm': 'Tối ưu cục bộ',     'Giải pháp': 'Simulated Annealing (Geom)',  'Hiệu quả (Accuracy/F1)': f'{sa_geom_fit*100:.2f}% (Macro-F1)',          'Khả năng giải thích': 'Trọng số lớp tường minh',      'Chi phí tính toán': 'Trung bình',        'Rủi ro lâm sàng': 'Cần tinh chỉnh nhiệt độ ban đầu'},
    {'Nhóm': 'Tối ưu cục bộ',     'Giải pháp': 'Simulated Annealing (Lin)',   'Hiệu quả (Accuracy/F1)': f'{sa_lin_fit*100:.2f}% (Macro-F1)',           'Khả năng giải thích': 'Trọng số lớp tường minh',      'Chi phí tính toán': 'Trung bình',        'Rủi ro lâm sàng': 'Làm nguội tuyến tính giảm dần'},
    {'Nhóm': 'Tối ưu tiến hóa',   'Giải pháp': 'Genetic Algorithm (GA)',      'Hiệu quả (Accuracy/F1)': f'{ga_best_fit*100:.2f}% (Macro-F1)',           'Khả năng giải thích': 'Trọng số lớp tối ưu',         'Chi phí tính toán': f'Cao (Pop {POP_SIZE} x {GENERATIONS})', 'Rủi ro lâm sàng': 'Hội tụ toàn cục, độ tin cậy cao'},
    {'Nhóm': 'Học máy cổ điển',   'Giải pháp': 'KNN (k=3, Scaled)',          'Hiệu quả (Accuracy/F1)': f'{acc_knn*100:.2f}% / {f1_knn_macro*100:.2f}%', 'Khả năng giải thích': 'Thấp (Khoảng cách điểm)',     'Chi phí tính toán': 'Chậm khi test (O(Nd))','Rủi ro lâm sàng': 'Nhạy cảm với nhiễu và ngoại lai'},
    {'Nhóm': 'Học máy cổ điển',   'Giải pháp': 'Decision Tree (depth=6)',     'Hiệu quả (Accuracy/F1)': f'{acc_dt*100:.2f}% / {f1_dt_macro*100:.2f}%',  'Khả năng giải thích': 'Trực quan (Cây quyết định)',   'Chi phí tính toán': 'Nhanh',              'Rủi ro lâm sàng': 'Dễ bị overfitting nếu cây quá sâu'},
    {'Nhóm': 'Học máy cổ điển',   'Giải pháp': 'Logistic Regression',         'Hiệu quả (Accuracy/F1)': f'{lr_test_acc*100:.2f}% / {f1_lr_macro*100:.2f}%', 'Khả năng giải thích': 'Tốt (Odds Ratio e^beta)',    'Chi phí tính toán': 'Cực nhanh',          'Rủi ro lâm sàng': 'Không bắt được tương tác phi tuyến'},
    {'Nhóm': 'Học máy cổ điển',   'Giải pháp': 'Support Vector Machine (RBF)','Hiệu quả (Accuracy/F1)': f'{svm_test_acc*100:.2f}% / {svm_test_f1*100:.2f}%', 'Khả năng giải thích': 'Hộp đen ranh giới phi tuyến', 'Chi phí tính toán': 'Trung bình',        'Rủi ro lâm sàng': 'Khó giải thích trọng số'},
    {'Nhóm': 'Học sâu (Deep)',    'Giải pháp': 'Keras MLP (class_weight)',    'Hiệu quả (Accuracy/F1)': f'{eval_res[1]*100:.2f}% / {f1_mlp_macro*100:.2f}%', 'Khả năng giải thích': 'Hộp đen (Rất thấp)',        'Chi phí tính toán': f'Cao ({stopped_epoch} epochs)', 'Rủi ro lâm sàng': 'Bác sĩ khó tin tưởng khi ra quyết định mổ'},
    {'Nhóm': 'Ensemble (TỐT NHẤT)','Giải pháp':'Random Forest Pipeline',      'Hiệu quả (Accuracy/F1)': f'{acc_rf*100:.2f}% / {f1_rf_macro*100:.2f}%',  'Khả năng giải thích': 'Khá (Gini Feature Imp)',       'Chi phí tính toán': 'Trung bình (Song song)', f'Rủi ro lâm sàng': f'Cực thấp, Recall Lớp 3 đạt {cm[2,2]/np.sum(cm[2])*100:.2f}%'}
])
print(all_solutions_df[['Giải pháp', 'Hiệu quả (Accuracy/F1)', 'Khả năng giải thích', 'Rủi ro lâm sàng']].to_string(index=False))

print("\nĐỀ XUẤT GIẢI PHÁP CUỐI CÙNG:")
print(f"-> Kiến trúc lai ghép (Hybrid Clinical Decision Support System):")
print(f"   Sử dụng Random Forest Pipeline làm bộ phân loại lõi (Acc {acc_rf*100:.2f}%, Macro-F1 {f1_rf_macro*100:.2f}%)")
print(f"   kết hợp Hệ chuyên gia luật IF-THEN ở tầng giải trình để cung cấp chuỗi suy diễn lùi cho bác sĩ.")

# 8b. Đóng gói Pipeline tốt nhất thành file .joblib [B08.4]
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', best_rf)
])
final_pipeline.fit(X_train, y_train)
joblib.dump(final_pipeline, 'DT69_123000609_BestModel.joblib')
print(f"\n✓ Đã đóng gói mô hình tối ưu: DT69_123000609_BestModel.joblib ({acc_rf*100:.2f}% Test Accuracy)")

# 8c. Viết hàm dự đoán cho 1 mẫu mới [B08.4]
def predict_single_sample(sample_dict):
    df_in = pd.DataFrame([sample_dict])
    pred = final_pipeline.predict(df_in)[0]
    proba = final_pipeline.predict_proba(df_in)[0]
    names = {1: 'Normal (Bình thường)', 2: 'Suspect (Nghi ngờ)', 3: 'Pathologic (Bệnh lý)'}
    return {
        'prediction_code': int(pred),
        'prediction_label': names[pred],
        'probabilities': {
            'Normal': round(float(proba[0]), 4),
            'Suspect': round(float(proba[1]), 4),
            'Pathologic': round(float(proba[2]), 4)
        }
    }

test_res = predict_single_sample(X_test.iloc[0].to_dict())
print(f"\nThử nghiệm dự đoán 1 ca bệnh mới:")
print(f"-> Dự đoán: {test_res['prediction_label']}")
print(f"-> Xác suất chi tiết: {test_res['probabilities']}")

# 8d. Xuất bộ thông số chuẩn xác thực nghiệm phục vụ đồng bộ tự động 100%
metrics_export = {
    "student": {
        "name": "Nguyễn Quách Phú Tài",
        "mssv": "123000609",
        "class": "23CT113",
        "project": "DT69",
        "seed": 99055,
        "sample_frac": 0.90,
        "sample_count": int(len(df_sample)),
        "train_count": int(len(X_train)),
        "test_count": int(len(X_test)),
        "cv_folds": 7,
        "ga_pop": 20,
        "ga_mut": 0.10,
        "knowledge_coverage": "25 / 40 (62.5%)"
    },
    "eda": {
        "class_leakage_acc": round(acc_leak, 4),
        "clean_baseline_acc": round(acc_clean, 4),
        "dummy_acc": round(dummy_acc, 4),
        "dummy_f1": round(dummy_f1, 4)
    },
    "expert_system": {
        "threshold_astv": round(th_astv, 2),
        "threshold_altv": round(th_altv, 2),
        "threshold_mstv": round(th_mstv, 2),
        "accuracy": round(rule_acc, 4),
        "f1_macro": round(f1_rules_macro, 4)
    },
    "search": {
        "bfs_nodes": int(res_bfs[2]),
        "bfs_acc": round(res_bfs[1], 4),
        "bfs_features": [top8_names[i] for i in res_bfs[0]],
        "greedy_h1_nodes": int(res_greedy_h1[2]),
        "greedy_h1_acc": round(res_greedy_h1[1], 4),
        "greedy_h1_features": [top8_names[i] for i in res_greedy_h1[0]],
        "greedy_h2_nodes": int(res_greedy_h2[2]),
        "greedy_h2_acc": round(res_greedy_h2[1], 4),
        "greedy_h2_features": [top8_names[i] for i in res_greedy_h2[0]],
        "astar_nodes": int(res_astar[2]),
        "astar_acc": round(res_astar[1], 4),
        "astar_features": [top8_names[i] for i in res_astar[0]],
        "cost_efficiency_ratio": int(res_bfs[2] // res_astar[2])
    },
    "optimization": {
        "initial_unweighted_f1": round(default_f1, 4),
        "hc_f1": round(hc_best_fit, 4),
        "hc_stuck_pct": round(hc_stuck_cnt / 20 * 100, 1),
        "hc_best_w": [round(float(x), 2) for x in hc_best_w],
        "sa_geom_f1": round(sa_geom_fit, 4),
        "sa_geom_w": [round(float(x), 2) for x in sa_geom_w],
        "sa_lin_f1": round(sa_lin_fit, 4),
        "sa_lin_w": [round(float(x), 2) for x in sa_lin_w],
        "ga_f1": round(ga_best_fit, 4),
        "ga_best_w": [round(float(x), 2) for x in ga_best_w]
    },
    "classical_ml": {
        "logreg_acc": round(lr_test_acc, 4),
        "logreg_f1": round(f1_lr_macro, 4),
        "odds_ratios_top3": [
            {"feature": r['Feature'], "beta": round(float(r['Coefficient (beta)']), 3), "odds_ratio": round(float(r['Odds Ratio (e^beta)']), 3)}
            for _, r in coef_df.head(3).iterrows()
        ],
        "knn_best_k": 3,
        "knn_scaled_acc": round(acc_knn, 4),
        "knn_f1": round(f1_knn_macro, 4),
        "knn_max_diff_pct": 0.85,
        "tree_best_depth": 6,
        "tree_cv_acc": round(sc_tree_d6, 4),
        "tree_test_acc": round(acc_dt, 4),
        "tree_f1": round(f1_dt_macro, 4),
        "svm_cv_acc": round(svm_cv_acc, 4),
        "svm_test_acc": round(svm_test_acc, 4),
        "svm_f1": round(svm_test_f1, 4)
    },
    "mlp": {
        "test_acc": round(float(eval_res[1]), 4),
        "test_loss": round(float(eval_res[0]), 4),
        "f1_macro": round(f1_mlp_macro, 4),
        "stopped_epoch": int(stopped_epoch),
        "stability_3seeds_mean": round(float(np.mean(keras_seed_scores)), 4),
        "stability_3seeds_std": round(float(np.std(keras_seed_scores)), 4),
        "manual_calc": {
            "x1": 0.73,
            "x2": 0.43,
            "h1": round(float(h1_val), 4),
            "h2": round(float(h2_val), 4),
            "softmax_probs": [round(float(p), 4) for p in probs],
            "loss_cce": round(float(loss_val), 4),
            "delta_out": [round(float(d), 4) for d in delta_out]
        }
    },
    "best_model_rf": {
        "params": grid_rf.best_params_,
        "cv7_mean": round(float(np.mean(cv7_scores)), 4),
        "cv7_std": round(float(np.std(cv7_scores)), 4),
        "test_acc": round(acc_rf, 4),
        "test_f1_macro": round(f1_rf_macro, 4),
        "test_precision_macro": round(float(classification_report(y_test, y_pred_rf, output_dict=True)['macro avg']['precision']), 4),
        "test_recall_macro": round(float(classification_report(y_test, y_pred_rf, output_dict=True)['macro avg']['recall']), 4),
        "roc_auc_macro": round(roc_auc_macro, 4),
        "confusion_matrix": cm.tolist(),
        "cm_details": {
            "normal_correct": int(cm[0, 0]), "normal_total": int(np.sum(cm[0])),
            "suspect_correct": int(cm[1, 1]), "suspect_total": int(np.sum(cm[1])),
            "pathologic_correct": int(cm[2, 2]), "pathologic_total": int(np.sum(cm[2]))
        },
        "top3_features": [
            {"feature": feat_imp.index[i], "importance": round(float(feat_imp.iloc[i]), 4)}
            for i in range(3)
        ],
        "ac_rank": int(list(feat_imp.index).index('AC') + 1),
        "ac_importance": round(float(feat_imp['AC']), 4)
    }
}

os.makedirs('scratch', exist_ok=True)
with open('scratch/exact_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_export, f, indent=2, ensure_ascii=False)
print("\n✓ Đã xuất bộ thông số chuẩn xác thực nghiệm: scratch/exact_metrics.json")


BẢNG SO SÁNH TOÀN DIỆN TẤT CẢ 14 GIẢI PHÁP ĐÃ XÂY DỰNG TRONG ĐỒ ÁN (ĐỘNG TỪ OUTPUT THỰC NGHIỆM)
                   Giải pháp Hiệu quả (Accuracy/F1)          Khả năng giải thích                           Rủi ro lâm sàng
       Dummy (most_frequent)        77.55% / 29.12%      Tầm thường (Đoán lớp 1) Bỏ sót 100% ca bệnh lý (Cực kỳ nguy hiểm)
      Luật IF-THEN (10 luật)        85.12% / 70.66%  Rất cao (Luật suy diễn lùi)  Bỏ sót các ca bệnh có tương tác phức tạp
               BFS (150 nút)        91.57% (CV Acc)    Rõ ràng (Tập 4 đặc trưng)             Bùng nổ tổ hợp, không thực tế
         Greedy (6 nút - MI)        86.93% (CV Acc)    Rõ ràng (Tập 4 đặc trưng)                 Dễ kẹt cực trị địa phương
     Greedy (6 nút - Custom)        86.93% (CV Acc)    Rõ ràng (Tập 4 đặc trưng)                 Dễ kẹt cực trị địa phương
           A* Search (6 nút)        86.93% (CV Acc) Tối ưu (Heuristic chấp nhận)               Phụ thuộc vào hàm Heuristic
 Hill Climbing (20 Restarts)      81.32% (M


✓ Đã đóng gói mô hình tối ưu: DT69_123000609_BestModel.joblib (95.04% Test Accuracy)

Thử nghiệm dự đoán 1 ca bệnh mới:
-> Dự đoán: Normal (Bình thường)
-> Xác suất chi tiết: {'Normal': 0.99, 'Suspect': 0.005, 'Pathologic': 0.005}

✓ Đã xuất bộ thông số chuẩn xác thực nghiệm: scratch/exact_metrics.json
